# Stage 13 — behavioral deviation and Relative Size Factor

Build short-window personal deviation and peer Relative Size Factor evidence for the later fraud risk score.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
BASE=Path('../outputs/model_ready/all_naive_12pct_200sims'); OUTPUT=Path('../outputs/deviation/all_naive_12pct_200sims'); OUTPUT.mkdir(parents=True,exist_ok=True)
h=pd.read_csv(BASE/'history_aware_sim_hour_features.csv',dtype={'sim':'string'},parse_dates=['hour_ts']); h=h.drop(columns=[c for c in h.columns if c.startswith('static_')],errors='ignore').sort_values(['sim','hour_ts']).reset_index(drop=True)
print('rows:',len(h),'SIMs:',h.sim.nunique())

In [ ]:
metrics=['events','calls','outgoing','incoming','duration_sum','unique_contacts','night_events']
for col in metrics:
    previous=h.groupby('sim',sort=False)[col].apply(lambda s:s.shift(1).expanding(min_periods=1).median()).reset_index(level=0,drop=True).to_numpy()
    h[f'{col}_personal_median_prev']=previous
    h[f'{col}_deviation_ratio']=h[col]/pd.Series(previous).replace(0,np.nan).to_numpy()
    h[f'{col}_deviation_delta']=h[col]-previous
h['has_personal_history']=h.groupby('sim').cumcount().gt(0)
print('rows with personal history:',int(h.has_personal_history.sum()))

In [ ]:
peer_metrics=['events','calls','outgoing','duration_sum','unique_contacts','night_events']
for col in peer_metrics:
    peer=h.groupby('hour_ts')[col].transform(lambda s:s.where(s>0).median())
    h[f'{col}_peer_median']=peer
    h[f'{col}_rsf']=h[col]/peer.replace(0,np.nan)
rsf=[c for c in h.columns if c.endswith('_rsf')]; dev=[c for c in h.columns if c.endswith('_deviation_ratio')]
h['rsf_high_count']=(h[rsf]>3).sum(axis=1); h['deviation_high_count']=(h[dev]>3).sum(axis=1); h['adaptive_evidence_count']=h.rsf_high_count+h.deviation_high_count
h.to_csv(OUTPUT/'behavioral_deviation_rsf_features.csv',index=False); print('saved feature rows:',len(h))

In [ ]:
summary=h.groupby('sim').agg(fraud=('fraud','max'),observed_hours=('hour_ts','nunique'),max_adaptive_evidence=('adaptive_evidence_count','max'),total_adaptive_evidence=('adaptive_evidence_count','sum'),max_events_rsf=('events_rsf','max'),max_duration_rsf=('duration_sum_rsf','max'),max_contacts_rsf=('unique_contacts_rsf','max'),max_events_deviation=('events_deviation_ratio','max'),max_duration_deviation=('duration_sum_deviation_ratio','max'),max_contacts_deviation=('unique_contacts_deviation_ratio','max')).reset_index()
summary['risk_evidence_layers']=(summary[['max_events_rsf','max_duration_rsf','max_contacts_rsf','max_events_deviation','max_duration_deviation','max_contacts_deviation']]>3).sum(axis=1)
summary.to_csv(OUTPUT/'sim_behavioral_deviation_summary.csv',index=False); print(summary.groupby('fraud')[['max_adaptive_evidence','total_adaptive_evidence','risk_evidence_layers']].median())

The first observed hour has no personal baseline. RSF compares a SIM with the peer median at the same hour. These are evidence indicators, not fraud decisions.